# Commonsense Knowledge Graph Construction Tutorial

Welcome to the Commonsense Knowledge Graph Construction project! This tutorial will guide you through setting up the project, configuring models, adding concepts, running the pipeline, and exporting results.

## 1. Setup

First, ensure you have installed the required dependencies. You can do this by running:

```bash
pip install -r ../requirements.txt
```

### API Keys

This project supports various LLM backends (Local, Groq, Nebula, OpenAI). 
To use them, you should set your API keys as environment variables. 
You can create a `.env` file in the root directory (or just set them in your shell/notebook).

Examples:
- `GROQ_API_KEY=your_groq_key`
- `OPENAI_API_KEY=your_openai_key`
- `NEBULA_API_KEY=your_nebula_key`

Let's check if we have any keys set (without printing them!):


In [2]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

key = os.getenv("NEBULA_API_KEY")
print("Starts with:", key[:4])
print("Ends with:", key[-4:])

Starts with: sk-1
Ends with: 778f


In [4]:
import os

print("GROQ_API_KEY set:", "GROQ_API_KEY" in os.environ)
print("OPENAI_API_KEY set:", "OPENAI_API_KEY" in os.environ)
print("NEBULA_API_KEY set:", "NEBULA_API_KEY" in os.environ)

GROQ_API_KEY set: False
OPENAI_API_KEY set: False
NEBULA_API_KEY set: True


## 2. Configuration

The models are defined in `config/models.yaml`. You can edit this file to add new models or change paths.
The properties and inputs are located in the `inputs` folder.

Let's look at the available models:

In [5]:
import yaml

with open("config/models.yaml", "r") as f:
    models = yaml.safe_load(f)

print(yaml.dump(models))

groq:
- model_path: meta-llama/llama-4-scout-17b-16e-instruct
  name: meta-llama-llama-4-scout-17b-16e-instruct
- model_path: openai/gpt-oss-120b
  name: openai-gpt-oss-120b
local:
- model_path: models/Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf
  name: llama31_8b_instruct
nebula:
- model_path: mercury.gpt-oss:20b
  name: gptoss_20b_standard
- model_path: mercury.gemma3:12b
  name: gemma3_12b_standard
- model_path: deepseek-r1:8b
  name: deepseekr1_8b_standard



## 3. Running the Pipeline

We have a convenient `ConstructionPipeline` class that handles the interaction with the LLMs.
You can run it directly from Python or use the `main.py` CLI.

### Using Python Class

Let's try to extract the property 'size' for the concept 'apple' using a local model or a mocked run if no model is available.

**Note:** Replace `model_name` with a valid model from your `models.yaml`. 
For this tutorial, we will verify the code structure effectively.


In [6]:
import sys
sys.path.append("..") # Add parent directory to path

from kg_constructors.pipeline import ConstructionPipeline

# Example configuration
concept = "apple"
property_name = "color"
template_name = "categorical" # Using emotions template as placeholder, ideally use one for size if available

# PLEASE CHANGE THIS TO A MODEL YOU HAVE ACCESS TO
# For example: 'gpt-3.5-turbo' if using OpenAI, or a local llama model name
model_name = "deepseekr1_8b_standard"

try:
    # Initialize the pipeline
    pipeline = ConstructionPipeline(
        concept=concept,
        prompt_template_name=template_name,
        property=property_name,
        model_name=model_name,
        ontology_path="tutorial_ontology.jsonl"
    )

    # Run the pipeline
    pipeline.run() # Uncomment to run if you have the model set up
    print("Pipeline initialized successfully!")

except Exception as e:
    print(f"Setup correct, but execution failed (likely due to missing model/key): {e}")

[INFO] [2026-03-02 17:39:28] Initialized ConstructionPipeline for concept: apple, model: deepseekr1_8b_standard
[INFO] [2026-03-02 17:39:28] Running pipeline for concept: apple with model: deepseekr1_8b_standard...
[INFO] [2026-03-02 17:39:28] 	...Running model with prompt:
	Given the concept of apple ,
provide a list of common colors for this concept.
Return only JSON with key:
  - color (string)...
[INFO] [2026-03-02 17:39:48] 	...Processed output with results: {'color': ['red', 'green', 'yellow']}
[INFO] [2026-03-02 17:39:48] 		...Updating ontology with results: {'color': ['red', 'green', 'yellow']}
[INFO] [2026-03-02 17:39:48] 	...Ontology updated with results.


Pipeline initialized successfully!


### Using CLI

You can also run the pipeline from the terminal:

```bash
python ../main.py --concept apple --model meta-llama-llama-4-scout-17b-16e-instruct --property size
```

Use `python ../main.py --help` to see all options.

## 4. Adding Concepts

To add new concepts, you typically create a new JSON or YAML file in `inputs/` or simply pass them programmatically as shown above.

- `inputs/emotions/` contains emotion-related concepts.
- `inputs/mscoco concepts/` contains objects from MSCOCO.

## 5. Exporting Results

The pipeline saves results to the specified `ontology_path` (default: `ontology.owl` or `ontology.jsonl`).

The format currently appends JSON objects for each run. Let's inspect a sample output structure:

```json
{
    "concept": "apple",
    "property": "size",
    "model": "llama3",
    "results": { "size": "small" }
}
```

You can load this file using pandas or standard json libraries for analysis.